# Final Integration and Construction of the Inverse Model

This notebook loads the five previously trained AISI 4140 property-prediction models (hardness, UTS, YS, elongation, reduction of area) and combines them into an **inverse model**: given a set of *desired* mechanical properties and a fixed section thickness, it searches over the heat-treatment process parameters (austenitizing temperature, tempering temperature, tempering time, quench medium) to find the recipe that is most likely to produce those properties.

**Workflow:**
1. Load the dataset and the trained forward-prediction pipelines.
2. Wrap the pipelines in a single `predict_properties()` forward-prediction function.
3. Define an optimization objective that scores how close a candidate process gets to the target properties.
4. Use `scipy.optimize.differential_evolution` to search for the best process parameters, for each available quench medium.
5. Export the results as CSV tables for the Power BI dashboard.

In [67]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Imports and Configuration

Import the standard data-science stack (`numpy`, `pandas`), `joblib` for loading the pickled model pipelines, and `differential_evolution` from `scipy.optimize`, which will later drive the inverse search.

Also defines `TemperingParameterTransformer`, a scikit-learn-compatible custom transformer that engineers the **tempering parameter** (a Hollomon–Jaffe-style combination of tempering temperature and time). This transformer mirrors the feature engineering baked into the trained pipelines, so it is kept here for reference/compatibility even though the pipelines already include it internally.

In [68]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import os
import glob
import json
import joblib
import numpy as np
import pandas as pd

from scipy.optimize import differential_evolution

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin


# ============================================================
# CUSTOM TRANSFORMER
# ============================================================

class TemperingParameterTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, C=20):
        self.C = C

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X["tempering_parameter"] = (
            (X["tempering_temp_C"] + 273.15)
            * (
                self.C
                + np.log10(X["tempering_time_hr"])
            )
        )

        return X

## 2. Project Paths

Mount Google Drive (again, in case this notebook is run standalone) and set up the project's folder structure: `data/` for the input dataset, `models/` for the trained pipelines, and `results/` for the exported inverse-model outputs.

In [69]:
# ============================================================
# 2. PROJECT PATH
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Heat_Treatment_Dashboard"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nModel directory:")
print(MODEL_DIR)

print("\nResults directory:")
print(RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory:
/content/drive/MyDrive/Heat_Treatment_Dashboard

Model directory:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models

Results directory:
/content/drive/MyDrive/Heat_Treatment_Dashboard/results


## 3. Load Dataset

Load the AISI 4140 heat-treatment dataset that the forward models were trained on. This same dataset is used later to determine realistic parameter ranges for the inverse search.

In [70]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

DATA_PATH = os.path.join(
    DATA_DIR,
    "aisi_4140_heat_treatment_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1100, 11)


,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct,source
0,845.0,25.0,oil,815.0,1.0,13.0,655.0,415.0,25.7,56.9,literature_anchor_annealed
1,845.0,25.0,oil,316.0,1.0,47.0,1551.0,1372.6,12.0,42.0,literature_anchor
2,845.0,25.0,oil,538.0,1.0,26.0,896.0,672.0,18.0,55.0,literature_anchor
3,851.7,50.0,oil,384.7,4.0,38.8,1212.1,1004.6,17.1,44.3,physics_model
4,834.4,100.0,air,401.7,4.0,37.4,1177.4,952.6,19.3,44.9,physics_model


## 4. Features and Targets

Define the process-parameter **input features** (austenitizing temperature, section thickness, quench medium, tempering temperature, tempering time) and the mechanical-property **targets** (hardness, UTS, YS, elongation, reduction of area) that the forward models predict and the inverse model will try to match.

In [71]:
# ============================================================
# 4. FEATURES AND TARGETS
# ============================================================

INPUT_FEATURES = [
    "austenitizing_temp_C",
    "section_thickness_mm",
    "quench_medium",
    "tempering_temp_C",
    "tempering_time_hr"
]

TARGETS = [
    "hardness_HRC",
    "UTS_MPa",
    "YS_MPa",
    "elongation_pct",
    "reduction_of_area_pct"
]

print("Input features:")
for feature in INPUT_FEATURES:
    print(" -", feature)

print("\nTarget properties:")
for target in TARGETS:
    print(" -", target)

Input features:
 - austenitizing_temp_C
 - section_thickness_mm
 - quench_medium
 - tempering_temp_C
 - tempering_time_hr

Target properties:
 - hardness_HRC
 - UTS_MPa
 - YS_MPa
 - elongation_pct
 - reduction_of_area_pct


## 5. Feature Engineering

Two engineered features feed the trained pipelines in addition to the raw process parameters:
- **`tempering_parameter`** — a Hollomon–Jaffe-style parameter combining tempering temperature and time into a single value.
- **`temperature_temp_time`** — a simple interaction term (temperature × time).

The first, standalone definition of `calculate_tempering_parameter` below is superseded by the version in the next cell (which adds a docstring and is used together with `prepare_features`); it's kept here as it was in the original analysis notebook.

In [72]:
def calculate_tempering_parameter(temp_C, time_hr):

    temp_K = temp_C + 273.15

    return temp_K * (
        20 + np.log10(time_hr)
    )

In [73]:
# ============================================================
# 5. FEATURE ENGINEERING
# ============================================================

def calculate_tempering_parameter(temp_C, time_hr):
    """
    Calculate the tempering parameter used by the trained models.
    """

    temp_K = temp_C + 273.15

    return temp_K * (
        20 + np.log10(time_hr)
    )

def prepare_features(input_data):

    input_data = input_data.copy()

    # Tempering parameter
    input_data["tempering_parameter"] = (
        (input_data["tempering_temp_C"] + 273.15)
        * (
            20
            + np.log10(input_data["tempering_time_hr"])
        )
    )

    # Temperature × time interaction
    input_data["temperature_temp_time"] = (
        input_data["tempering_temp_C"]
        * input_data["tempering_time_hr"]
    )

    return input_data


## 6. Locate the Trained Model Pipelines

Since the pipeline `.pkl` files may live anywhere under the mounted Google Drive, `find_model_file()` searches recursively for each filename and returns the first match, raising a clear error if a file can't be found.

In [74]:
# ============================================================
# 6. FIND MODEL PIPELINES
# ============================================================

def find_model_file(filename):

    matches = glob.glob(
        os.path.join(
            "/content/drive/MyDrive",
            "**",
            filename
        ),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} anywhere in Google Drive."
        )

    print(f"Found {filename}:")
    print(matches[0])

    return matches[0]

## 7. Model File Registry

Maps each target property to the filename of its corresponding trained scikit-learn pipeline (`.pkl`).

In [75]:
MODEL_FILES = {
    "hardness_HRC": "aisi4140_hardness_HRC_pipeline.pkl",
    "UTS_MPa": "aisi4140_UTS_MPa_pipeline.pkl",
    "YS_MPa": "aisi4140_YS_MPa_pipeline.pkl",
    "elongation_pct": "aisi4140_elongation_pct_pipeline.pkl",
    "reduction_of_area_pct": "aisi4140_reduction_of_area_pct_pipeline.pkl"
}

## 8. Load All Five Trained Models

Loads every pipeline listed in `MODEL_FILES` into the `pipelines` dictionary, keyed by target property name, so they can be used together for prediction.

In [76]:
# ============================================================
# LOAD ALL FIVE MODELS
# ============================================================

pipelines = {}

for target, filename in MODEL_FILES.items():

    path = find_model_file(filename)

    pipelines[target] = joblib.load(path)

    print(f"{target}: loaded successfully")

print("\nAll models loaded.")

Found aisi4140_hardness_HRC_pipeline.pkl:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models/aisi4140_hardness_HRC_pipeline.pkl
hardness_HRC: loaded successfully
Found aisi4140_UTS_MPa_pipeline.pkl:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models/aisi4140_UTS_MPa_pipeline.pkl
UTS_MPa: loaded successfully
Found aisi4140_YS_MPa_pipeline.pkl:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models/aisi4140_YS_MPa_pipeline.pkl
YS_MPa: loaded successfully
Found aisi4140_elongation_pct_pipeline.pkl:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models/aisi4140_elongation_pct_pipeline.pkl
elongation_pct: loaded successfully
Found aisi4140_reduction_of_area_pct_pipeline.pkl:
/content/drive/MyDrive/Heat_Treatment_Dashboard/models/aisi4140_reduction_of_area_pct_pipeline.pkl
reduction_of_area_pct: loaded successfully

All models loaded.


## 9. Model Validation (Sanity Check)

Runs a single sample row from the dataset (with engineered features added) through all five loaded pipelines and prints their predictions, as a quick sanity check that every model loaded correctly and produces reasonable output.

In [77]:
# ============================================================
# 7. MODEL VALIDATION
# ============================================================

sample = df[INPUT_FEATURES].iloc[[0]].copy()

# Add engineered features
sample["tempering_parameter"] = (
    (sample["tempering_temp_C"] + 273.15)
    * (
        20 + np.log10(sample["tempering_time_hr"])
    )
)

sample["temperature_temp_time"] = (
    sample["tempering_temp_C"]
    * sample["tempering_time_hr"]
)

print("Sample input:")
display(sample)

for target, pipeline in pipelines.items():

    prediction = pipeline.predict(sample)[0]

    print(
        f"{target}: {prediction:.4f}"
    )

Sample input:


,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr,tempering_parameter,temperature_temp_time
0,845.0,25.0,oil,815.0,1.0,21763.0,815.0


hardness_HRC: 13.4596
UTS_MPa: 677.6760
YS_MPa: 456.4534
elongation_pct: 24.6570
reduction_of_area_pct: 55.6654


## 10. Forward Prediction Function

`predict_properties()` is the single entry point for **forward** prediction: given raw process parameters, it builds a one-row DataFrame, applies `prepare_features()`, runs it through all five pipelines, and returns a dictionary of predicted mechanical properties. This function is reused inside the inverse-optimization objective below.

In [78]:
# ============================================================
# 8. FORWARD PREDICTION FUNCTION
# ============================================================

def predict_properties(
    austenitizing_temp_C,
    section_thickness_mm,
    quench_medium,
    tempering_temp_C,
    tempering_time_hr
):

    input_data = pd.DataFrame({
        "austenitizing_temp_C": [austenitizing_temp_C],
        "section_thickness_mm": [section_thickness_mm],
        "quench_medium": [quench_medium],
        "tempering_temp_C": [tempering_temp_C],
        "tempering_time_hr": [tempering_time_hr]
    })

    input_data = prepare_features(input_data)

    predictions = {}

    for target, pipeline in pipelines.items():

        predictions[target] = pipeline.predict(
            input_data
        )[0]

    return predictions

## 11. Example Forward Prediction

Quick example call to `predict_properties()` with a representative set of process parameters, to confirm the forward pipeline works end-to-end.

In [79]:
example_prediction = predict_properties(
    austenitizing_temp_C=850,
    section_thickness_mm=50,
    quench_medium="oil",
    tempering_temp_C=500,
    tempering_time_hr=2
)

example_prediction

{'hardness_HRC': np.float64(28.55070940579455),
 'UTS_MPa': np.float64(950.5724102819918),
 'YS_MPa': np.float64(698.0708523890659),
 'elongation_pct': np.float64(22.16538650481568),
 'reduction_of_area_pct': np.float64(50.6076105399008)}

## 12. Inverse Search Space (Fixed Bounds)

An initial, hard-coded guess at reasonable bounds for the process parameters. These are superseded by the data-driven ranges computed in the next cell, which derives the bounds directly from the min/max values observed in the dataset — kept here for reference on the originally assumed process envelope.

In [80]:
# ============================================================
# 10. INVERSE SEARCH SPACE
# ============================================================

AUSTENITIZING_RANGE = (828, 872)

THICKNESS_RANGE = (13, 150)

TEMPERING_TEMP_RANGE = (202.5, 815)

TEMPERING_TIME_RANGE = (0.25, 8)

## 13. Inverse Search Space (Data-Driven Bounds)

Recomputes the search bounds for each process parameter directly from the dataset's observed min/max values, and collects the list of available quench media. These are the ranges actually used by the optimizer below.

In [81]:
AUSTENITIZING_RANGE = (
    df["austenitizing_temp_C"].min(),
    df["austenitizing_temp_C"].max()
)

THICKNESS_RANGE = (
    df["section_thickness_mm"].min(),
    df["section_thickness_mm"].max()
)

TEMPERING_TEMP_RANGE = (
    df["tempering_temp_C"].min(),
    df["tempering_temp_C"].max()
)

TEMPERING_TIME_RANGE = (
    df["tempering_time_hr"].min(),
    df["tempering_time_hr"].max()
)

QUENCH_MEDIA = sorted(
    df["quench_medium"].unique()
)

print("Austenitizing:", AUSTENITIZING_RANGE)
print("Thickness:", THICKNESS_RANGE)
print("Tempering temperature:", TEMPERING_TEMP_RANGE)
print("Tempering time:", TEMPERING_TIME_RANGE)
print("Quench media:", QUENCH_MEDIA)

Austenitizing: (828.0, 871.9)
Thickness: (13.0, 150.0)
Tempering temperature: (202.5, 815.0)
Tempering time: (0.5, 4.0)
Quench media: ['air', 'oil', 'water']


## 14. Normalized Error Function

It computes the **sum of squared relative errors** between the predicted and target properties. Using *relative* (percentage-style) error rather than raw absolute error puts all five properties — which have very different scales (e.g. hardness in HRC vs. UTS in MPa) — on a comparable footing, so no single property dominates the optimization purely because of its units.

In [82]:
def normalized_error(predictions, target_properties):
    """
    Compute a scale-free error between predicted and target mechanical
    properties, so properties with very different units/magnitudes
    (e.g. hardness in HRC vs. UTS in MPa) contribute comparably to the
    optimization objective.

    For each property, the relative error is (prediction - target) / target.
    The total error is the sum of squared relative errors across all
    target properties.
    """

    total_error = 0.0

    for prop, target_value in target_properties.items():

        predicted_value = predictions[prop]

        relative_error = (predicted_value - target_value) / target_value

        total_error += relative_error ** 2

    return total_error

## 15. Inverse Optimization Objective

`inverse_objective()` is the scalar function that `differential_evolution` will minimize. For a candidate set of process parameters `x` (austenitizing temperature, tempering temperature, tempering time), it runs the forward model via `predict_properties()` and scores the result against the desired properties using `normalized_error()`. Lower values mean the candidate process is predicted to produce properties closer to the target.

*(Note: this cell and the next define the exact same function — the duplicate was left in place as in the original notebook; only the second definition is actually in effect.)*

In [83]:
# ============================================================
# 12. INVERSE OPTIMIZATION OBJECTIVE
# ============================================================

def inverse_objective(
    x,
    section_thickness,
    quench_medium,
    target_properties
):

    (
        austenitizing_temp,
        tempering_temp,
        tempering_time
    ) = x

    predictions = predict_properties(
        austenitizing_temp_C=austenitizing_temp,
        section_thickness_mm=section_thickness,
        quench_medium=quench_medium,
        tempering_temp_C=tempering_temp,
        tempering_time_hr=tempering_time
    )

    return normalized_error(
        predictions,
        target_properties
    )

In [84]:
# ============================================================
# 12. INVERSE OPTIMIZATION
# ============================================================

def inverse_objective(
    x,
    section_thickness,
    quench_medium,
    target_properties
):

    (
        austenitizing_temp,
        tempering_temp,
        tempering_time
    ) = x

    predictions = predict_properties(
        austenitizing_temp_C=austenitizing_temp,
        section_thickness_mm=section_thickness,
        quench_medium=quench_medium,
        tempering_temp_C=tempering_temp,
        tempering_time_hr=tempering_time
    )

    return normalized_error(
        predictions,
        target_properties
    )

## 16. Inverse Search: Find a Heat-Treatment Cycle

`find_heat_treatment()` runs the actual inverse search. For **each quench medium**, it uses `differential_evolution` to search the (austenitizing temperature, tempering temperature, tempering time) space for the combination that minimizes `inverse_objective` — i.e. best matches the desired properties. It then re-runs the forward model on the best parameters found, records the result, and returns a DataFrame of one candidate solution per quench medium, sorted from best (lowest error) to worst.

In [85]:
# ============================================================
# 13. FIND HEAT-TREATMENT CYCLE
# ============================================================

def find_heat_treatment(
    target_properties,
    section_thickness,
    maxiter=100,
    popsize=10
):

    results = []

    # Only these variables are optimized
    bounds = [
        AUSTENITIZING_RANGE,
        TEMPERING_TEMP_RANGE,
        TEMPERING_TIME_RANGE
    ]

    for quench_medium in QUENCH_MEDIA:

        print(
            f"Searching for quench medium: "
            f"{quench_medium}"
        )

        result = differential_evolution(

            inverse_objective,

            bounds=bounds,

            args=(
                section_thickness,
                quench_medium,
                target_properties
            ),

            maxiter=maxiter,
            popsize=popsize,
            seed=42,
            polish=True
        )

        (
            austenitizing_temp,
            tempering_temp,
            tempering_time
        ) = result.x

        # Generate final predictions
        predictions = predict_properties(
            austenitizing_temp_C=austenitizing_temp,
            section_thickness_mm=section_thickness,
            quench_medium=quench_medium,
            tempering_temp_C=tempering_temp,
            tempering_time_hr=tempering_time
        )

        results.append({

            "austenitizing_temp_C":
                austenitizing_temp,

            "section_thickness_mm":
                section_thickness,

            "quench_medium":
                quench_medium,

            "tempering_temp_C":
                tempering_temp,

            "tempering_time_hr":
                tempering_time,

            "objective_error":
                result.fun,

            **predictions

        })

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        "objective_error"
    ).reset_index(drop=True)

    return results_df

## 17. Test the Inverse Model

Define a target set of desired mechanical properties to search for.

In [86]:
# ============================================================
# 14. TEST INVERSE MODEL
# ============================================================

desired_properties = {

    "hardness_HRC": 40,

    "UTS_MPa": 1350,

    "YS_MPa": 1150,

    "elongation_pct": 12,

    "reduction_of_area_pct": 35
}

## 18. Run the Inverse Search

For a component with a 50 mm section thickness, run `find_heat_treatment()` against the desired properties defined above and display the resulting candidate heat-treatment cycles (one per quench medium, best first).

In [87]:
# Section thickness of the component
section_thickness = 50  # mm

inverse_results = find_heat_treatment(
    target_properties=desired_properties,
    section_thickness=section_thickness
)

display(inverse_results)

Searching for quench medium: air
Searching for quench medium: oil
Searching for quench medium: water


,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr,objective_error,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
0,869.852917,50,water,307.847469,3.104167,0.074070,46.254750,1501.998512,1250.388521,14.039317,35.735238
1,869.781764,50,oil,306.634599,3.315472,0.078187,46.285259,1513.089882,1266.313424,14.015181,35.759840
2,828.945782,50,air,203.892309,0.614339,0.122531,41.275346,1361.991800,1092.912015,15.823687,39.622021


## 19. Best Solution

Pull out the single best candidate (lowest `objective_error`) from the results.

In [88]:
# ============================================================
# 15. BEST SOLUTION
# ============================================================

best_solution = inverse_results.iloc[0].copy()

best_solution_df = pd.DataFrame(
    [best_solution]
)

display(best_solution_df)

,austenitizing_temp_C,section_thickness_mm,quench_medium,tempering_temp_C,tempering_time_hr,objective_error,hardness_HRC,UTS_MPa,YS_MPa,elongation_pct,reduction_of_area_pct
0,869.852917,50,water,307.847469,3.104167,0.07407,46.25475,1501.998512,1250.388521,14.039317,35.735238


## 20. Target vs. Predicted Properties

Build a side-by-side comparison table of the desired target properties against the properties predicted for the best solution, along with the absolute error for each property, to assess how well the inverse search matched the goal.

In [89]:
# ============================================================
# 16. TARGET VS PREDICTION
# ============================================================

target_vs_prediction = pd.DataFrame({

    "Property": [
        "Hardness",
        "UTS",
        "YS",
        "Elongation",
        "Reduction of Area"
    ],

    "Target": [
        desired_properties["hardness_HRC"],
        desired_properties["UTS_MPa"],
        desired_properties["YS_MPa"],
        desired_properties["elongation_pct"],
        desired_properties["reduction_of_area_pct"]
    ],

    "Predicted": [
        best_solution["hardness_HRC"],
        best_solution["UTS_MPa"],
        best_solution["YS_MPa"],
        best_solution["elongation_pct"],
        best_solution["reduction_of_area_pct"]
    ]
})

target_vs_prediction["Absolute_Error"] = (
    target_vs_prediction["Predicted"]
    - target_vs_prediction["Target"]
).abs()

display(target_vs_prediction)

,Property,Target,Predicted,Absolute_Error
0,Hardness,40,46.254750,6.254750
1,UTS,1350,1501.998512,151.998512
2,YS,1150,1250.388521,100.388521
3,Elongation,12,14.039317,2.039317
4,Reduction of Area,35,35.735238,0.735238


## 21. Export Results for Power BI

Save three CSV tables to the `results/` folder for the Power BI dashboard:
- **`powerbi_best_heat_treatment.csv`** — the single best recommended process.
- **`powerbi_candidate_heat_treatments.csv`** — all candidate processes (one per quench medium).
- **`powerbi_target_vs_prediction.csv`** — target vs. predicted property comparison.

In [90]:
# ============================================================
# 17. EXPORT POWER BI TABLES
# ============================================================

BEST_PATH = os.path.join(
    RESULTS_DIR,
    "powerbi_best_heat_treatment.csv"
)

CANDIDATE_PATH = os.path.join(
    RESULTS_DIR,
    "powerbi_candidate_heat_treatments.csv"
)

TARGET_PATH = os.path.join(
    RESULTS_DIR,
    "powerbi_target_vs_prediction.csv"
)

best_solution_df.to_csv(
    BEST_PATH,
    index=False
)

inverse_results.to_csv(
    CANDIDATE_PATH,
    index=False
)

target_vs_prediction.to_csv(
    TARGET_PATH,
    index=False
)

print("Power BI files saved.")

Power BI files saved.
